# Knowledge Extractor - Extracción de Entidades ATC

Este notebook procesa documentos PDF de fraseología aeronáutica para extraer entidades y relaciones usando NLP con Ollama.

## Importación de librerías

Importamos todas las dependencias necesarias para el procesamiento:
- `ollama`: Para interactuar con el modelo de lenguaje local
- `json`: Para manejar estructuras de datos JSON
- `re`: Para expresiones regulares
- `os`: Para operaciones de sistema de archivos
- `pypdf`: Para leer y extraer texto de archivos PDF

In [ ]:
import ollama
import json
import re
import os
from pypdf import PdfReader

import numpy as np
from sentence_transformers import SentenceTransformer
import nltk

# Descargar datos necesarios para tokenización de oraciones
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

# Cargar el documento PDF con PyMuPDF (fitz) para extracción precisa
try:
    import fitz  # PyMuPDF
except ImportError:
    raise ImportError("PyMuPDF (fitz) es requerido para extracción precisa de texto. Instala con: pip install PyMuPDF")

/home/gabo/Personal/Universidad/04 - Cuarto Año/2do Semestre/Tesis/Proyecto/atc-alert-system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Función NER con Ollama y Contexto

Esta función utiliza el modelo de lenguaje Ollama para realizar Named Entity Recognition (NER) y extracción de relaciones en texto de control de tráfico aéreo, **ahora con contexto de entidades previas**.

**Características:**
- Extrae entidades aeronáuticas (aeródromos, altitudes, aeronaves, etc.)
- Identifica relaciones operativas entre entidades
- Usa un prompt especializado para dominio ATC
- **NUEVO: Mantiene consistencia con entidades extraídas en páginas anteriores**
- Devuelve resultados en formato JSON estructurado

**Parámetros:**
- `texto`: Texto de entrada para analizar
- `entidades_previas`: Lista de entidades extraídas en páginas anteriores (opcional)
- `modelo`: Modelo de Ollama a usar (default: "llama3")

**Mejoras implementadas:**
- Contexto de entidades previas para mantener consistencia
- Nuevo constraint que exige usar texto y etiquetas idénticos para entidades recurrentes
- Deduplicación automática de entidades acumuladas

In [3]:
def ner_con_ollama(texto, entidades_previas=None, modelo="llama3"):
    """Usa Ollama para NER con contexto de entidades previas"""

    prompt = """
        ATC NER & Relationship Extraction Prompt
        "Extract ALL instances of aeronautical entities and their operational relationships in the following JSON format:

        {
        "entities": [
            {
            "text": "the exact text from the source",
            "label": "the category of the entity",
            "context": "brief semantic description"
            }
        ],
        "relationships": [
            {
            "subject": "Entity A",
            "predicate": "The relationship or action connecting them",
            "object": "Entity B"
            }
        ]
        }

        [Specify Constraints]:

            Focus: Only consider entities and relationships related to Air Traffic Control (ATC) procedures, flight constraints (altitude, speed, heading, position), and facility responsibilities.
            Dynamic Discovery: Do not limit extraction to a fixed list. While you should prioritize standard categories (such as Aerodrome, Altitude, ARTCC, Fix/Position, Sector, and Tower), you must also extract any other domain-specific entity discovered in the text that impacts flight operations or airspace management.
            Relationship Type: Prioritize extracting "subject-predicate-object" triplets that define operational rules or constraints (e.g., "Arrivals - must use - CEDES STAR").
            Entity Consistency: Maintain strict consistency with previously extracted entities. Use identical text and labels for recurring entities. If you encounter an entity that matches a previously extracted one, use the exact same text and label rather than creating variations.
            Relationship Entity Validation: All entities referenced in relationships (both subject and object) MUST appear in the entities section of the current extraction OR be present in the previously extracted entities list. Do not create relationships with entities that have not been explicitly extracted as entities.
    """

    # Añadir entidades previas si existen
    if entidades_previas:
        prompt += "\n\n[Previously Extracted Entities]:\n"
        for entidad in entidades_previas:
            # Soporta entidades como dict incompleto, strings, o cualquier objeto printable
            if isinstance(entidad, dict):
                text = entidad.get("text", "")
                label = entidad.get("label", "Unknown")
            else:
                text = str(entidad)
                label = "Unknown"

            text = (text or "").strip()
            label = (label or "Unknown").strip()
            if not text:
                continue

            prompt += f"- {text} ({label})\n"

    prompt += "\n\n[Input Specification]:\n"
    prompt += (texto or "") + "\nJSON Output ONLY:\n"

    response = ollama.chat(model=modelo, messages=[
        {
            'role': 'user',
            'content': prompt
        },
    ])

    return response['message']['content']

## Extractor de JSON robusto

Esta función extrae JSON válido de texto que puede contener bloques de código o respuestas de LLM.

**Problema resuelto:** JSON no es un lenguaje regular, no se puede parsear con regex debido a anidación infinita.

**Solución:** Parser manual que:
- Busca balance correcto de llaves `{}` considerando strings y escapes
- Maneja bloques de código ```json``` o ```
- Ignora llaves dentro de strings
- Procesa caracteres de escape correctamente
- Soporta anidación arbitraria

**Ventajas sobre regex:**
- Maneja estructuras JSON complejas
- Evita falsos positivos por llaves en strings
- Soporta JSON multilinea correctamente

In [4]:
def extract_json(texto):
    """
    Extrae JSON de un texto usando un parser manual que maneja anidación correctamente.
    """
    
    def encontrar_json_por_llaves(texto):
        """
        Encuentra el JSON válido buscando el balance correcto de llaves {}
        """
        for i, char in enumerate(texto):
            if char == '{':
                # Encontrar el cierre de llave correspondiente
                balance = 1
                j = i + 1
                escape_next = False
                in_string = False
                string_char = None
                
                while j < len(texto) and balance > 0:
                    char_actual = texto[j]
                    
                    if not escape_next:
                        if char_actual == '\\' and in_string:
                            escape_next = True
                        elif char_actual in ['"', "'"] and not escape_next:
                            if not in_string:
                                in_string = True
                                string_char = char_actual
                            elif char_actual == string_char:
                                in_string = False
                                string_char = None
                        elif not in_string:
                            if char_actual == '{':
                                balance += 1
                            elif char_actual == '}':
                                balance -= 1
                    
                    if escape_next:
                        escape_next = False
                    
                    j += 1
                
                if balance == 0:
                    # Encontramos un JSON potencial
                    json_str = texto[i:j]
                    try:
                        return json.loads(json_str), j
                    except json.JSONDecodeError:
                        continue
        
        return None, -1
    
    # Primero intentar encontrar y eliminar bloques de código ```json o ```
    texto_limpio = texto
    patron_bloque = r'```(?:json)?\s*\n(.*?)\s*```'
    match_bloque = re.search(patron_bloque, texto, re.DOTALL)
    
    if match_bloque:
        # Extraer el contenido del bloque
        contenido_bloque = match_bloque.group(1).strip()
        resultado, pos = encontrar_json_por_llaves(contenido_bloque)
        if resultado:
            return resultado
    
    # Si no funciona con bloque, buscar en todo el texto
    resultado, pos = encontrar_json_por_llaves(texto_limpio)
    if resultado:
        return resultado
    
    return None


## Configuración inicial

Definimos las rutas y parámetros para el procesamiento:
- `doc_path`: Ruta al archivo PDF de entrada
- `output_dir`: Directorio base para archivos de salida
- `doc_name`: Nombre del documento derivado del filename

Esta configuración permite procesar diferentes documentos cambiando solo las variables iniciales.

In [5]:
def configure_output(doc_path, output_dir, model_name):
    # Configuración
    doc_name = doc_path.split("/")[-1].split(".")[0] + f"({model_name})"

    # Crear directorio de salida si no existe
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Creado directorio: {output_dir}")

    # Crear carpeta para el documento si no existe
    doc_dir = os.path.join(output_dir, doc_name)
    if not os.path.exists(doc_dir):
        os.makedirs(doc_dir)
        print(f"Creada carpeta para el documento: {doc_dir}")

    return doc_dir

## Función para acumular entidades desde una carpeta

Esta función permite **reconstruir un catálogo consolidado de entidades** a partir de los archivos JSON generados por el extractor (por ejemplo, `pagina_1.json`, `pagina_2.json`, etc.). Es útil para:
- Reutilizar entidades ya extraídas como **contexto inicial** (`initial_context`) al procesar otro documento
- Exportar un **inventario único** de entidades detectadas en un procesamiento previo

**Qué hace:**
- Recorre todos los archivos `pagina_*.json` dentro de una carpeta
- Lee las entidades extraídas en cada archivo
- Soporta tanto resultados a nivel de página como a nivel de oraciones (detectando `sentence_results`)
- **Deduplica** entidades por texto normalizado (lowercase + strip)
- Conserva la **primera aparición** de cada entidad como referencia

**Salida:**
- Retorna un diccionario `accumulated` con estructura:
  - `normalized_text -> { text, label, context, first_seen_page, source_file }`

**Notas:**
- Si un archivo tiene un formato inesperado o está corrupto, se registra el error y el procesamiento continúa con el siguiente archivo.


In [6]:
def extract_accumulated_entities_from_folder(folder_path):
    """
    Lee todos los JSON de una carpeta y extrae todas las entidades acumuladas
    (útil para exportar catálogo consolidado o usar como initial_context)
    """
    from pathlib import Path
    import json
    
    folder = Path(folder_path)
    if not folder.is_dir():
        raise ValueError(f"La carpeta no existe: {folder_path}")
    
    accumulated = {}  # normalized_text -> entity_info
    
    for json_file in sorted(folder.glob("pagina_*.json")):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Detectar si es sentence-level o page-level
            chunks = data.get('sentence_results') or [data]  # lista de chunks
            
            for chunk in chunks:
                ner_data = chunk.get('ner', {})
                if not isinstance(ner_data, dict) or 'entities' not in ner_data:
                    continue
                
                for entity in ner_data['entities']:
                    if not isinstance(entity, dict) or 'text' not in entity:
                        continue
                    
                    text = entity['text']
                    normalized = (text or "").lower().strip()
                    if not normalized:
                        continue
                    
                    # Conservar la primera aparición (más completa)
                    if normalized not in accumulated:
                        accumulated[normalized] = {
                            'text': text,
                            'label': entity.get('label', 'Unknown'),
                            'context': entity.get('context', ''),
                            'first_seen_page': data.get('page_number', '?'),
                            'source_file': data.get('source_file', json_file.name),
                        }
                    else:
                        # Podríamos actualizar frecuencia o mantener la primera aparición
                        pass
                        
        except Exception as e:
            print(f"Error procesando {json_file.name}: {e}")
    
    print(f"Entidades únicas acumuladas desde {folder}: {len(accumulated)}")
    return accumulated

## Procesamiento principal del PDF con Contexto Acumulado

Esta celda ejecuta el flujo completo de procesamiento **mejorado con contexto**:

1. **Extracción de texto**: Lee cada página del PDF y extrae el texto (con soporte para márgenes personalizados)
2. **NER con Ollama**: Aplica reconocimiento de entidades usando el modelo LLM con contexto de entidades previas
3. **Acumulación de entidades**: Mantiene una lista creciente de entidades únicas encontradas
4. **Deduplicación**: Evita repetir entidades idénticas basadas en texto normalizado
5. **Parseo de JSON**: Extrae el JSON estructurado de la respuesta del LLM con validación
6. **Guardado**: Crea archivos JSON por página con:
   - `texto_original`: Texto extraído del PDF
   - `ner`: Entidades y relaciones extraídas (o texto plano si falla)
   - `contexto_entidades_usadas`: Número de entidades de contexto disponibles

**Mejoras clave:**
- **Consistencia**: El LLM recibe contexto de entidades previas para mantener nomenclatura consistente
- **Deduplicación**: Sistema automático para evitar entidades duplicadas
- **Métricas**: Registro de cuántas entidades de contexto se usaron por página
- **Validación JSON**: Verificación automática de la estructura del JSON extraído
- **Márgenes personalizables**: Extracción de texto de áreas específicas del PDF

**Parámetro margins:**
- Permite especificar un rectángulo de extracción como `(left, bottom, right, top)`
- Las coordenadas son distancias desde los bordes de la página en puntos (pt)
- `left`: distancia desde el borde izquierdo
- `bottom`: distancia desde el borde inferior
- `right`: distancia desde el borde derecho
- `top`: distancia desde el borde superior
- Si es `None`, extrae el texto completo de la página

**Salida:** Archivos `pagina_N.json` en el directorio del documento con información de contexto y márgenes

**Manejo de errores:** Si el parseo JSON falla la validación, guarda la respuesta original del LLM como fallback.

In [ ]:
def kex(
    doc_path,
    doc_dir,
    model_name,
    embed_model_name="sentence-transformers/all-MiniLM-L6-v2",
    context_top_k=50,
    context_threshold=0.1,
    page_embed_max_chars=4000,
    granularity="page",  # "page" o "sentence"
    start_page=1,  # página de inicio (1-indexed)
    initial_context=None,  # nueva: lista de entidades iniciales para bootstrap
    tokenizer_language="english",  # idioma para tokenización de NLTK
    margins=None,  # márgenes para extracción de texto: (left, bottom, right, top) o None para extraer toda la página
):

    def _normalize_entity_text(text):
        return (text or "").lower().strip()

    def _entity_to_embed_text(entity):
        # Usamos text + label + context para dar más señal semántica
        text = entity.get("text", "")
        label = entity.get("label", "Unknown")
        context = entity.get("context", "")
        return f"{text} ({label}) ({context})".strip()

    def _cosine_sim_matrix(a, b):
        # a: (d,), b: (n,d)
        denom = (np.linalg.norm(a) * np.linalg.norm(b, axis=1))
        denom = np.where(denom == 0, 1e-12, denom)
        return (b @ a) / denom

    def _validate_ner_json(ner_json):
        """Valida que el JSON extraído tenga la estructura esperada"""
        if not isinstance(ner_json, dict):
            return False, "El JSON no es un diccionario"
        
        # Validar entities
        if "entities" not in ner_json:
            return False, "Falta la clave 'entities'"
        
        if not isinstance(ner_json["entities"], list):
            return False, "'entities' no es una lista"
        
        for i, entity in enumerate(ner_json["entities"]):
            if not isinstance(entity, dict):
                return False, f"Entidad {i} no es un diccionario"
            if "text" not in entity:
                return False, f"Entidad {i} no tiene 'text'"
            if "label" not in entity:
                return False, f"Entidad {i} no tiene 'label'"
        
        # Validar relationships (opcional)
        if "relationships" in ner_json:
            if not isinstance(ner_json["relationships"], list):
                return False, "'relationships' no es una lista"
            
            for i, rel in enumerate(ner_json["relationships"]):
                if not isinstance(rel, dict):
                    return False, f"Relación {i} no es un diccionario"
                required_rel_keys = ["subject", "predicate", "object"]
                for key in required_rel_keys:
                    if key not in rel:
                        return False, f"Relación {i} no tiene '{key}'"
        
        return True, "JSON válido"

    def select_context_entities(texto_chunk, entities_list, entities_embeddings, top_k, threshold):
        if not entities_list:
            return []

        texto = (texto_chunk or "")
        if page_embed_max_chars and len(texto) > page_embed_max_chars:
            texto = texto[:page_embed_max_chars]

        page_emb = embed_model.encode(texto, normalize_embeddings=False)
        sims = _cosine_sim_matrix(page_emb, entities_embeddings)

        idx_sorted = np.argsort(-sims)
        selected = []
        for idx in idx_sorted[:top_k]:
            sim = float(sims[idx])
            if sim < threshold:
                break
            selected.append(entities_list[idx])

        return selected

    def _process_text_chunks(chunks, entities_by_norm, entity_order_norm, entity_embeddings):
        """Procesa una lista de chunks (páginas u oraciones) y acumula entidades/embeddings."""
        for chunk_text, page_num, chunk_idx in chunks:
            # Seleccionar contexto por embeddings (antes de llamar al LLM)
            entities_list_for_context = [entities_by_norm[n] for n in entity_order_norm]
            if entity_embeddings is None:
                context_entities = []
            else:
                context_entities = select_context_entities(
                    chunk_text,
                    entities_list_for_context,
                    entity_embeddings,
                    context_top_k,
                    context_threshold,
                )

            # Aplicar NER usando Ollama con contexto seleccionado
            resultado_ner = ner_con_ollama(chunk_text, context_entities, model_name)

            # Extraer JSON del resultado
            ner_json = extract_json(resultado_ner)

            # Validar el JSON extraído
            if ner_json:
                is_valid, validation_msg = _validate_ner_json(ner_json)
                if not is_valid:
                    print(f"Chunk {page_num}-{chunk_idx}: JSON inválido - {validation_msg}")
                    ner_json = None

            if ner_json:
                print(f"Chunk {page_num}-{chunk_idx}: JSON extraído y validado correctamente")

                # Acumular nuevas entidades (evitando duplicados)
                if "entities" in ner_json and isinstance(ner_json["entities"], list):
                    new_entities = []
                    for nueva_entidad in ner_json["entities"]:
                        if not isinstance(nueva_entidad, dict):
                            continue
                        norm = _normalize_entity_text(nueva_entidad.get("text"))
                        if not norm:
                            continue

                        if norm not in entities_by_norm:
                            entities_by_norm[norm] = nueva_entidad
                            entity_order_norm.append(norm)
                            new_entities.append(nueva_entidad)

                    # Actualizar embeddings solo para las nuevas entidades
                    if new_entities:
                        new_emb_texts = [_entity_to_embed_text(e) for e in new_entities]
                        new_embs = embed_model.encode(new_emb_texts, normalize_embeddings=False)
                        new_embs = np.atleast_2d(new_embs)

                        if entity_embeddings is None:
                            entity_embeddings = new_embs
                        else:
                            entity_embeddings = np.vstack([entity_embeddings, new_embs])

                print(f"  - Entidades acumuladas: {len(entities_by_norm)}")
                print(f"  - Entidades en contexto (seleccionadas): {len(context_entities)}")
            else:
                print(f"Chunk {page_num}-{chunk_idx}: No se pudo extraer JSON válido")

            yield {
                "chunk_text": chunk_text,
                "ner": ner_json,
                "llm_output": resultado_ner,
                "context": {
                    "embedding_model": embed_model_name,
                    "top_k": context_top_k,
                    "threshold": context_threshold,
                    "page_embed_max_chars": page_embed_max_chars,
                    "contexto_entidades_usadas": len(context_entities),
                    "contexto_entidades_seleccionadas": [
                        {"text": e.get("text"), "label": e.get("label"), "context": e.get("context", "")} for e in context_entities
                    ],
                    "entidades_acumuladas_total": len(entities_by_norm),
                },
            }
  
    doc = fitz.open(doc_path)
    numero_paginas = doc.page_count

    # Validar start_page
    if start_page < 1 or start_page > numero_paginas:
        raise ValueError(f"start_page debe estar entre 1 y {numero_paginas}")

    # Validar márgenes si se proporcionan
    if margins is not None:
        if not isinstance(margins, (list, tuple)) or len(margins) != 4:
            raise ValueError("margins debe ser una tupla/lista de 4 elementos: (left, bottom, right, top)")
        if any(not isinstance(m, (int, float)) for m in margins):
            raise ValueError("Todos los valores de margins deben ser numéricos")
        print(f"Usando márgenes personalizados: {margins}")

    print(f"Procesando páginas {start_page} a {numero_paginas} ({numero_paginas - start_page + 1} páginas) con granularity='{granularity}' y tokenizer_language='{tokenizer_language}'...")

    # Embedding model
    embed_model = SentenceTransformer(embed_model_name)

    # Estructuras acumuladas (inicializadas con initial_context si existe)
    entities_by_norm = {}
    entity_order_norm = []
    entity_embeddings = None

    # Procesar contexto inicial si se proporciona
    if initial_context:
        print(f"Cargando contexto inicial con {len(initial_context)} entidades...")
        for entidad in initial_context:
            if not isinstance(entidad, dict):
                continue
            norm = _normalize_entity_text(entidad.get("text"))
            if not norm:
                continue
            if norm not in entities_by_norm:
                entities_by_norm[norm] = entidad
                entity_order_norm.append(norm)

        # Generar embeddings para el contexto inicial
        if entities_by_norm:
            initial_emb_texts = [_entity_to_embed_text(e) for e in entities_by_norm.values()]
            entity_embeddings = embed_model.encode(initial_emb_texts, normalize_embeddings=False)
            print(f"Contexto inicial procesado: {len(entities_by_norm)} entidades con embeddings")

    # Procesar cada página y crear JSON
    for i in range(start_page - 1, numero_paginas):  # 0-indexed range
        page = doc[i]
        
        # Extraer texto con márgenes usando PyMuPDF (respeta los límites)
        if margins is not None:
            left, bottom, right, top = margins
            
            # PyMuPDF usa coordenadas desde esquina superior izquierda (0,0)
            page_width = page.rect.width
            page_height = page.rect.height
            
            # Convertir márgenes a coordenadas absolutas para PyMuPDF
            # margins = (left, bottom, right, top) donde:
            # left, right: distancia desde borde izquierdo
            # bottom, top: distancia desde borde inferior/superior
            
            # Para PyMuPDF (coordenadas desde top-left):
            x0 = left  # distancia desde izquierda
            y0 = top   # distancia desde arriba (PyMuPDF usa top-left origin)
            x1 = page_width - right  # distancia desde izquierda hasta borde derecho menos margen derecho
            y1 = page_height - bottom  # distancia desde arriba hasta borde inferior menos margen inferior
            
            # Crear rectángulo de recorte
            clip_rect = fitz.Rect(x0, y0, x1, y1)
            
            # Extraer texto solo del área especificada
            texto_original = page.get_text(clip=clip_rect)
            print(f"Página {i+1}: extraído texto de área recortada ({x0:.1f}, {y0:.1f}, {x1:.1f}, {y1:.1f})")
        else:
            # Extraer texto completo de la página
            texto_original = page.get_text()
            print(f"Página {i+1}: extraído texto completo")
        

        if granularity == "sentence":
            # Dividir en oraciones con el idioma especificado
            sentences = nltk.sent_tokenize(texto_original, language=tokenizer_language)
            chunks = [(sent, i + 1, idx) for idx, sent in enumerate(sentences)]
        else:  # "page"
            chunks = [(texto_original, i + 1, 0)]

        # Procesar chunks de la página - CORREGIDO: actualizar entity_embeddings
        page_results = []
        for result in _process_text_chunks(chunks, entities_by_norm, entity_order_norm, entity_embeddings):
            page_results.append(result)
            # Actualizar entity_embeddings si se modificó en _process_text_chunks
            if result["ner"] and "entities" in result["ner"]:
                # Verificar si se agregaron nuevas entidades en este chunk
                for entidad in result["ner"]["entities"]:
                    norm = _normalize_entity_text(entidad.get("text"))
                    if norm in entities_by_norm and norm not in entity_order_norm[:-len(result["ner"]["entities"])]:
                        # Se agregaron entidades nuevas, regenerar embeddings
                        entities_list = [entities_by_norm[n] for n in entity_order_norm]
                        emb_texts = [_entity_to_embed_text(e) for e in entities_list]
                        entity_embeddings = embed_model.encode(emb_texts, normalize_embeddings=False)
                        break

        # Guardar como archivo JSON
        pagina_data = {
            "texto_original": texto_original,
            "granularity": granularity,
            "tokenizer_language": tokenizer_language,
            "margins": margins,
            "sentence_results" if granularity == "sentence" else "page_results": page_results,
        }

        filename = f"pagina_{i+1}.json"
        filepath = os.path.join(doc_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(pagina_data, f, ensure_ascii=False, indent=2)

        print(f"Creado archivo: {filename}")

    # Cerrar documento
    doc.close()

    print(f"\nProceso completado. Archivos guardados en: {doc_dir}")
    print(f"Total de entidades únicas acumuladas: {len(entities_by_norm)}")

In [9]:
# Ejemplo: extraer entidades acumuladas de una carpeta y usarlas como initial_context
# folder_path = "output/4_kex_output/ICAO Standard Phraseology(deepseek-r1:7b)"
# accumulated_entities = extract_accumulated_entities_from_folder(folder_path)
# initial_context_list = list(accumulated_entities.values())

In [ ]:
# Configuración
doc_path = "../docs/ICAO Standard Phraseology.pdf"
output_dir = "output/test"

doc_dir = configure_output(doc_path, output_dir, "llama3.2")

margins = (34, 76, 1, 33)  # valores que estabas usando
kex(doc_path, doc_dir, "llama3.2", granularity="sentence", tokenizer_language="english", margins=margins)

Usando márgenes personalizados: (34, 76, 1, 33)
Procesando páginas 1 a 20 (20 páginas) con granularity='sentence' y tokenizer_language='english'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 955.09it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Página 1: extraído texto de área recortada (34.0, 33.0, 594.0, 766.0)
 
ICAO Standard Phraseology 
A Quick Reference Guide for  
Commercial Air Transport Pilots 
Communication error is the biggest causal factor 
in both level busts and runway incursions in 
Europe. 
This 
document 
aims 
to 
provide 
Commercial Air Transport (CAT) pilots and other 
pilots flying IFR within controlled airspace with a 
quick 
reference 
guide 
to 
commonly 
used 
radiotelephony 
(RTF) 
phrases 
that 
may 
be 
encountered during a routine CAT flight in 
European Airspace. 

[(' \nICAO Standard Phraseology \nA Quick Reference Guide for  \nCommercial Air Transport Pilots \nCommunication error is the biggest causal factor \nin both level busts and runway incursions in \nEurope.', 1, 0), ('This \ndocument \naims \nto \nprovide \nCommercial Air Transport (CAT) pilots and other \npilots flying IFR within controlled airspace with a \nquick \nreference \nguide \nto \ncommonly \nused \nradiotelephony \n(RTF) \nphr